In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm

# device detection for CUDA Apple MPS or CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

# ensure models folder exists
os.makedirs("models", exist_ok=True)

In [ ]:

try:
    trainLoader  # if this exists from 01_data_prep
    valLoader
    testLoader
    dataset  # for class names
    print("Using dataloaders from memory")
except NameError:
    # Fallback: recreate dataset and dataloaders quickly
    from torchvision import datasets, transforms
    from torch.utils.data import random_split, DataLoader

    dataDir = "data"
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    fullDataset = datasets.ImageFolder(dataDir, transform=transform)
    dsSize = len(fullDataset)
    trainSize = int(0.7 * dsSize)
    valSize = int(0.15 * dsSize)
    testSize = dsSize - trainSize - valSize
    trainDataset, valDataset, testDataset = random_split(fullDataset, [trainSize, valSize, testSize])
    valDataset.dataset.transform = transform
    testDataset.dataset.transform = transform

    batchSize = 16  # safe default for M3
    trainLoader = DataLoader(trainDataset, batch_size=batchSize, shuffle=True, num_workers=0)
    valLoader = DataLoader(valDataset, batch_size=batchSize, shuffle=False, num_workers=0)
    testLoader = DataLoader(testDataset, batch_size=batchSize, shuffle=False, num_workers=0)
    dataset = fullDataset
    print(f"Recreated dataloaders  train {len(trainDataset)} val {len(valDataset)} test {len(testDataset)}")

In [ ]:

model = models.densenet121(pretrained=True)
numFeatures = model.classifier.in_features
model.classifier = nn.Linear(numFeatures, 4)  # 4 classes
model = model.to(device)

# Optionally freeze features at first
freezeFeatures = True
if freezeFeatures:
    for param in model.features.parameters():
        param.requires_grad = False

print(model)


In [ ]:
criterion = nn.CrossEntropyLoss()
# Only train parameters that require grad
trainableParams = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainableParams, lr=1e-4, weight_decay=1e-4)
# optional lr scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, verbose=True)

# training settings
numEpochs = 20
saveEveryNEpochs = 5  # periodic checkpoint
bestValLoss = float("inf")
startEpoch = 0

In [ ]:
for epoch in range(startEpoch, numEpochs):
    epochStart = time.time()
    model.train()
    runningLoss = 0.0
    totalTrain = 0
    correctTrain = 0

    for inputs, labels in tqdm(trainLoader, desc=f"Train epoch {epoch+1}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        runningLoss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correctTrain += (preds == labels).sum().item()
        totalTrain += labels.size(0)

    trainLoss = runningLoss / totalTrain
    trainAcc = correctTrain / totalTrain

    # validation
    model.eval()
    valRunningLoss = 0.0
    totalVal = 0
    correctVal = 0
    with torch.no_grad():
        for inputs, labels in tqdm(valLoader, desc=f"Val epoch {epoch+1}"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            valRunningLoss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correctVal += (preds == labels).sum().item()
            totalVal += labels.size(0)

    valLoss = valRunningLoss / totalVal
    valAcc = correctVal / totalVal

    # scheduler step
    scheduler.step(valLoss)

    # save periodic checkpoint
    if (epoch + 1) % saveEveryNEpochs == 0:
        ckptPath = f"models/densenetAllEpoch{epoch+1}.pth"
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": valLoss
        }, ckptPath)
        print(f"Saved checkpoint {ckptPath}")

    # save best model by validation loss
    if valLoss < bestValLoss:
        bestValLoss = valLoss
        bestPath = "models/densenetAllBest.pth"
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": valLoss
        }, bestPath)
        print(f"Saved best model {bestPath}  valLoss {valLoss:.4f}")

    epochTime = time.time() - epochStart
    print(f"Epoch {epoch+1}/{numEpochs}  trainLoss {trainLoss:.4f} trainAcc {trainAcc:.4f}  valLoss {valLoss:.4f} valAcc {valAcc:.4f}  time {epochTime:.1f}s")

In [ ]:
resumePath = "models/densenetAllEpoch10.pth"  # change as needed
if os.path.exists(resumePath):
    checkpoint = torch.load(resumePath, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    startEpoch = checkpoint["epoch"]
    print(f"Resumed from {resumePath} at epoch {startEpoch}")

# Load best model for evaluation
bestPath = "models/densenetAllBest.pth"
if os.path.exists(bestPath):
    checkpoint = torch.load(bestPath, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print(f"Loaded best model from {bestPath}  valLoss {checkpoint.get('val_loss'):.4f}")
else:
    print("Best model not found. Train first.")